In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Exakte Simulation mit Qiskit SDK Primitives

<details>
<summary><b>Package versions</b></summary>

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese Versionen oder neuere zu verwenden.

```
qiskit[all]~=2.3.0
```
</details>
Die Referenz-Primitives im Qiskit SDK führen lokale Statevector-Simulationen durch. Diese Simulationen unterstützen keine
Modellierung von Geräterauschen, eignen sich aber gut zum schnellen Prototypisieren von Algorithmen, bevor du fortgeschrittenere Simulationstechniken ([mit Qiskit Aer](./simulate-stabilizer-circuits)) oder echte Geräte ([Qiskit Runtime Primitives](primitives)) nutzt.

Das Estimator Primitive kann Erwartungswerte von Circuits berechnen, und das Sampler Primitive kann
aus den Ausgabeverteilungen von Circuits sampeln.

Die folgenden Abschnitte zeigen, wie du die Referenz-Primitives verwendest, um deinen Workflow lokal auszuführen.

## Den Referenz-Estimator verwenden
Die Referenzimplementierung von `EstimatorV2` in `qiskit.primitives`, die auf einem lokalen Statevector-Simulator läuft,
ist die Klasse [`StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator). Sie akzeptiert Circuits, Observablen und Parameter als Eingaben und gibt die lokal berechneten Erwartungswerte zurück.

Der folgende Code bereitet die Eingaben vor, die in den nachfolgenden Beispielen verwendet werden. Der erwartete Eingabetyp für die
Observablen ist [`qiskit.quantum_info.SparsePauliOp`](../api/qiskit/qiskit.quantum_info.SparsePauliOp). Beachte, dass
der Circuit im Beispiel parametrisiert ist, du aber den Estimator auch auf nicht-parametrisierte Circuits anwenden kannst.

> **Note:** Jeder Circuit, der an einen Estimator übergeben wird, darf **keine** **Messungen** enthalten.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# circuit for which you want to obtain the expected value
circuit = QuantumCircuit(2)
circuit.ry(Parameter("theta"), 0)
circuit.h(0)
circuit.cx(0, 1)
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/5b41a52d-8f15-4ce4-b3f6-effd91946d9c-0.svg" alt="Output of the previous code cell" />

In [2]:
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# observable(s) whose expected values you want to compute

observable = SparsePauliOp(["II", "XX", "YY", "ZZ"], coeffs=[1, 1, -1, 1])

# value(s) for the circuit parameter(s)
parameter_values = [[0], [np.pi / 6], [np.pi / 2]]

> **Tip:** Der Qiskit Runtime Primitives-Workflow erfordert, dass Circuits und Observablen so transformiert werden, dass sie nur Anweisungen verwenden, die vom QPU unterstützt werden (auch als *Instruction Set Architecture (ISA)*-Circuits und Observablen bezeichnet). Die Referenz-Primitives akzeptieren weiterhin abstrakte Anweisungen, da sie auf lokalen Statevector-Simulationen basieren. Das Transpilieren des Circuits kann jedoch in Bezug auf die Circuit-Optimierung dennoch vorteilhaft sein.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(circuit)
>   isa_observable = observable.apply_layout(isa_circuit.layout)
>   ```

### Estimator initialisieren
Instanziiere einen [`qiskit.primitives.StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator).

In [3]:
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

### Ausführen und Ergebnisse abrufen
Dieses Beispiel verwendet nur einen Circuit (vom Typ [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit)) und eine
Observable.

Führe die Schätzung aus, indem du die Methode [`StatevectorEstimator.run`](../api/qiskit/qiskit.primitives.StatevectorEstimator#run) aufrufst, die eine Instanz eines [`PrimitiveJob`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveJob)-Objekts zurückgibt. Du kannst die Ergebnisse aus dem Job (als [`qiskit.primitives.PrimitiveResult`](../api/qiskit/qiskit.primitives.PrimitiveResult)-Objekt)
mit der Methode [`qiskit.primitives.PrimitiveJob.result`](../api/qiskit/qiskit.primitives.PrimitiveJob#result) abrufen.

In [4]:
job = estimator.run([(circuit, observable, parameter_values)])
result = job.result()
print(f" > Result class: {type(result)}")

 > Result class: <class 'qiskit.primitives.containers.primitive_result.PrimitiveResult'>


#### Get the expected value from the result

The primitives result outputs an array of [`PubResult`](/docs/api/qiskit/qiskit.primitives.PubResult#pubresult) objects, where each item of the array is a `PubResult` object that contains in its data the array of evaluations corresponding to every circuit-observable combination in the PUB.

To retrieve the expectation values and metadata for the first (and in this case, only) circuit evaluation, we must access the evaluation [`data`](/docs/api/qiskit/qiskit.primitives.PubResult#data) for PUB 0:

In [5]:
print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

 > Expectation value: [4.         3.73205081 2.        ]
 > Metadata: {'target_precision': 0.0, 'circuit_metadata': {}}


#### Den Erwartungswert aus dem Ergebnis auslesen
Die Ausgabe der Primitives enthält ein Array von [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult)-Objekten, wobei jedes Element des Arrays ein `PubResult`-Objekt ist, das in seinen Daten das Array der Auswertungen für jede Circuit-Observable-Kombination im PUB enthält.

Um die Erwartungswerte und Metadaten für die erste (und in diesem Fall einzige) Circuit-Auswertung abzurufen, musst du auf die Auswertungs-[`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#data) für PUB 0 zugreifen:

In [6]:
# Estimate expectation values for two PUBs, both with 0.05 precision.
precise_job = estimator.run(
    [(circuit, observable, parameter_values)], precision=0.05
)

For a full example, see the [Primitives examples](primitives-examples#estimator-examples) page.

## Use the reference Sampler

The reference implementations of `SamplerV2` in `qiskit.primitives` is the [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler) class. It takes circuits and parameters as inputs and returns the results from sampling from the output probability distributions as a quasi-probability distribution of output states.

The following code prepares the inputs used in the examples that follow. Note that
these examples run a single parametrized circuit, but you can also run the Sampler
on non-parametrized circuits.

In [7]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg" alt="Output of the previous code cell" />

### Estimator-Ausführungsoptionen festlegen
Standardmäßig führt der Referenz-Estimator eine exakte Statevector-Berechnung basierend auf der
Klasse [`quantum_info.Statevector`](../api/qiskit/qiskit.quantum_info.Statevector) durch.
Dies kann jedoch angepasst werden, um den Effekt des Sampling-Overheads (auch bekannt als „Shot-Rauschen") einzubeziehen.

Der Estimator akzeptiert ein `precision`-Argument, das die Fehlerbalken angibt, auf die die
Primitive-Implementierung bei Erwartungswert-Schätzungen abzielen soll. Dies ist der Sampling-Overhead und wird ausschließlich in der `.run()`-Methode definiert. Damit kannst du die Option bis auf PUB-Ebene fein abstimmen.

In [8]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

Ein vollständiges Beispiel findest du auf der Seite [Primitives-Beispiele](primitives-examples#estimator-examples).

## Den Referenz-Sampler verwenden
Die Referenzimplementierung von `SamplerV2` in `qiskit.primitives` ist die Klasse [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler). Sie akzeptiert Circuits und Parameter als Eingaben und gibt die Ergebnisse des Sampelns aus den Ausgabe-Wahrscheinlichkeitsverteilungen als Quasi-Wahrscheinlichkeitsverteilung der Ausgabezustände zurück.

Der folgende Code bereitet die Eingaben vor, die in den nachfolgenden Beispielen verwendet werden. Beachte, dass
diese Beispiele einen einzelnen parametrisierten Circuit ausführen, du den Sampler aber auch auf nicht-parametrisierte Circuits anwenden kannst.

In [9]:
# execute 1 circuit with Sampler
job = sampler.run([circuit])
pub_result = job.result()[0]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg)

> **Note:** Jeder Quantenschaltkreis, der an einen Sampler übergeben wird, **muss** Messungen enthalten.

> **Tip:** Der Qiskit Runtime Primitives-Workflow erfordert, dass Circuits so transformiert werden, dass sie nur Anweisungen verwenden, die vom QPU unterstützt werden (auch als ISA-Circuits bezeichnet). Die Referenz-Primitives akzeptieren weiterhin abstrakte Anweisungen, da sie auf lokalen Statevector-Simulationen basieren. Das Transpilieren des Circuits kann jedoch in Bezug auf die Circuit-Optimierung dennoch vorteilhaft sein.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(qc)
>   ```

### `SamplerV2` initialisieren
Instanziiere [`qiskit.primitives.StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler):

In [10]:
from qiskit.transpiler import generate_preset_pass_manager

# create two circuits
circuit1 = circuit.copy()
circuit2 = circuit.copy()

# transpile circuits
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit1 = pm.run(circuit1)
isa_circuit2 = pm.run(circuit2)
# execute 2 circuits using Sampler
job = sampler.run([(isa_circuit1), (isa_circuit2)])
pub_result_1 = job.result()[0]
pub_result_2 = job.result()[1]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


### Ausführen und Ergebnisse abrufen

In [11]:
# Define quantum circuit with 2 qubits
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw()

        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 

In [12]:
# Transpile circuit
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit = pm.run(circuit)
# Run using sampler
result = sampler.run([circuit]).result()
# Access result data for PUB 0
data_pub = result[0].data
# Access bitstring for the classical register "meas"
bitstrings = data_pub.meas.get_bitstrings()
print(f"The number of bitstrings is: {len(bitstrings)}")
# Get counts for the classical register "meas"
counts = data_pub.meas.get_counts()
print(f"The counts are: {counts}")

The number of bitstrings is: 1024
The counts are: {'11': 515, '00': 509}


Primitives akzeptieren mehrere PUBs als Eingaben, und jeder PUB erhält sein eigenes Ergebnis. Daher kannst du verschiedene Circuits mit unterschiedlichen Parameter-/Observable-Kombinationen ausführen und die PUB-Ergebnisse abrufen:

In [13]:
# Sample two circuits at 128 shots each.
sampler.run([isa_circuit1, isa_circuit2], shots=128)
# Sample two circuits at different amounts of shots. The "None"s are necessary
# as placeholders
# for the lack of parameter values in this example.
sampler.run([(isa_circuit1, None, 123), (isa_circuit2, None, 456)])

For a full example, see the [Primitives examples](./primitives-examples#sampler-examples) page.
## Next steps

<Admonition type="tip" title="Recommendations">
  - For higher-performance simulation that can handle larger circuits, or to incorporate noise models into your simulation, see [Exact and noisy simulation with Qiskit Aer primitives](simulate-with-qiskit-aer).
  - To learn how to use Quantum Composer for simulation, see the [IBM Quantum Composer](/docs/guides/composer) guide.
  - Read the [Qiskit Estimator API](/docs/api/qiskit/1.4/qiskit.primitives.Estimator) reference.
  - Read the [Qiskit Sampler API](/docs/api/qiskit/1.4/qiskit.primitives.Sampler) reference.
  - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>